# Track Analysis: Corners, Braking & Acceleration Zones

This notebook automatically detects and visualizes track features from your telemetry data.

## What You'll Find Here

- **Corner Detection**: Automatically identifies corners from GPS curvature data, including direction (L/R), apex location, and approximate radius
- **Corner Map**: Interactive GPS map showing detected corners with apex markers
- **Braking Zone Detection**: Identifies where heavy braking occurs, averaged across your fastest laps
- **Acceleration Zone Detection**: Identifies throttle application zones, with gear-change gaps merged
- **Track Segment Visualization**: Color-coded GPS map showing braking (red), corners (orange), and acceleration (green) zones

## How It Works

1. **Corner Detection**: Computes track curvature from GPS coordinates and identifies sustained high-curvature sections
2. **Zone Averaging**: Analyzes laps within 103% of your best lap time to find consistent braking/acceleration patterns
3. **Segment Creation**: Combines corners with braking/accel zones into a complete track segmentation

## Using Your Own Data

To analyze your own data:

1. **Run the file picker cell** below to display the upload widget
2. **Drag and drop** your `.xrk` or `.xrz` file onto the upload button (or click to browse)
3. **Tune detection parameters** (optional): The corner detection has parameters you can adjust:
   - `threshold=0.006`: Curvature threshold (~167m radius). Lower = detect gentler corners
   - `min_corner_length=15`: Minimum corner length in samples
   - `min_gap=80`: Merge same-direction corners within this distance (meters)
4. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`) for zone detection

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

# Import core libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.graph_objects as go

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions
from motorsports_data_notebook import (
    show_fig,
    get_best_lap,
    compute_lap_distance,
    identify_corners,
    FileUpload,
)
from motorsports_data_notebook.helpers import (
    identify_zones_single_lap,
    average_zones_across_laps,
    merge_accel_zones_by_time,
    TrackSegment,
    create_track_segments,
)

# File picker - upload your own .xrk/.xrz file or use the sample data
file_upload = FileUpload(default_file="CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")
file_upload.display()

In [ ]:
# Load the data file
log = aim_xrk(file_upload.get_file_data())

In [ ]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()

# Add derived columns
channels["speed_kmh"] = channels["GPS Speed"] * 3.6

In [ ]:
# Load laps and compute lap times
laps = log.laps.to_pandas()
laps["lap_time"] = pd.to_timedelta(laps["end_time"] - laps["start_time"], unit="ms")

# Compute distance_m for each lap and add to channels
# Distance resets at the start of each lap
channels["distance_m"] = 0.0

for idx, lap in laps.iterrows():
    lap_mask = (channels["timecodes"] >= lap["start_time"]) & (
        channels["timecodes"] <= lap["end_time"]
    )
    lap_indices = channels.index[lap_mask]

    if len(lap_indices) > 0:
        lap_timecodes = channels.loc[lap_indices, "timecodes"]
        lap_speed = channels.loc[lap_indices, "GPS Speed"]
        distance_values = compute_lap_distance(lap_timecodes.values, lap_speed.values)
        channels.loc[lap_indices, "distance_m"] = distance_values

laps.style.format(
    {"lap_time": lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"}
)

In [ ]:
# Best lap extraction
best_lap = get_best_lap(laps)
start_ts = best_lap["start_time"]
end_ts = best_lap["end_time"]
# Use < for end_ts to exclude the first sample of the next lap (where distance resets to 0)
lap_channels = channels.query(f"timecodes >= @start_ts and timecodes < @end_ts").copy()

In [ ]:
# Identify corners directly from GPS coordinates
# identify_corners handles GPS->XY conversion, curvature computation, and corner detection
corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.006,  # Tuned for GPS noise - ~167m radius threshold
    min_corner_length=15,  # Reduced to catch shorter corners
    min_gap=80,  # Merge same-direction corners within 80m
)

print(f"Found {len(corners)} corners:")
for c in corners:
    print(
        f"  {c.name} ({c.direction}): {c.start_dist:.0f}m - {c.end_dist:.0f}m (apex at {c.apex_dist:.0f}m, radius ~{c.radius:.0f}m)"
    )

In [ ]:
# Visualize corners on GPS map with markers
fig = go.Figure()

# Plot track colored by speed
fig.add_trace(
    go.Scattermapbox(
        lat=lap_channels["GPS Latitude"],
        lon=lap_channels["GPS Longitude"],
        mode="markers",
        marker=dict(
            size=5,
            color=lap_channels["speed_kmh"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Speed (km/h)"),
        ),
        name="Track",
    )
)

# Add corner apex markers
for corner in corners:
    apex_idx = corner.apex_idx
    fig.add_trace(
        go.Scattermapbox(
            lat=[lap_channels["GPS Latitude"].iloc[apex_idx]],
            lon=[lap_channels["GPS Longitude"].iloc[apex_idx]],
            mode="markers+text",
            marker=dict(size=15, color="red"),
            text=[corner.name],
            textposition="top right",
            textfont=dict(size=12, color="red"),
            name=corner.name,
        )
    )

fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(
            lat=lap_channels["GPS Latitude"].mean(), lon=lap_channels["GPS Longitude"].mean()
        ),
        zoom=14,
    ),
    title="Detected Corners",
    showlegend=False,
    width=800,
    height=600,
)

show_fig(fig)

In [ ]:
# Identify braking and acceleration zones averaged over top laps
# Get laps within 103% of best lap time
valid_laps = laps[laps["lap_time"] > pd.Timedelta(0)].copy()
best_lap_time = valid_laps["lap_time"].min()
threshold_time = best_lap_time * 1.03
top_laps = valid_laps[valid_laps["lap_time"] <= threshold_time]

print(f"Best lap time: {best_lap_time}")
print(f"103% threshold: {threshold_time}")
print(
    f"Using {len(top_laps)} laps within 103% of best (out of {len(valid_laps)} valid laps) for zone averaging"
)

# Collect zones from each top lap
all_braking_zones = []
all_accel_zones = []

for idx, lap in top_laps.iterrows():
    lap_start = lap["start_time"]
    lap_end = lap["end_time"]
    lap_data = channels.query(f"timecodes >= @lap_start and timecodes <= @lap_end").copy()

    if len(lap_data) < 10:
        continue

    # Use pre-computed distance_m from channels
    braking, accel = identify_zones_single_lap(
        lap_data["distance_m"].values,
        lap_data["BrakePress"].values,
        lap_data["PPS"].values,
        lap_data["GPS Speed"].values,  # Speed in m/s for time-based gear change detection
    )
    all_braking_zones.append(braking)
    all_accel_zones.append(accel)

# Average zones across laps (use 50% threshold - at least half of laps must agree)
braking_zones, accel_zones = average_zones_across_laps(
    all_braking_zones,
    all_accel_zones,
    track_length=lap_channels["distance_m"].max(),
    resolution=1.0,
    threshold=0.5,
)

# Post-process: merge acceleration zones separated by short time gaps (gear changes)
accel_zones = merge_accel_zones_by_time(
    accel_zones,
    braking_zones,
    lap_channels["distance_m"].values,
    lap_channels["GPS Speed"].values,
    max_gap_time=1.5,  # 1.5 seconds to bridge gear changes
)

print(
    f"Found {len(braking_zones)} braking zones and {len(accel_zones)} acceleration zones (averaged over {len(top_laps)} laps)"
)

In [ ]:
# Create fixed segment definitions combining corners with braking/accel zones
track_length = lap_channels["distance_m"].iloc[-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"Created {len(segments)} track segments:")
for seg in segments:
    print(
        f"  [{seg.segment_type:12}] {seg.name:20} : {seg.start_dist:6.0f}m - {seg.end_dist:6.0f}m"
    )

In [ ]:
# Visualize track segments on GPS map

fig = go.Figure()

# Create a distance-to-index mapping for GPS coordinates
distance_arr = lap_channels["distance_m"].values
lat_arr = lap_channels["GPS Latitude"].values
lon_arr = lap_channels["GPS Longitude"].values


def get_indices_for_range(start_dist, end_dist):
    """Get indices corresponding to a distance range."""
    mask = (distance_arr >= start_dist) & (distance_arr <= end_dist)
    return np.where(mask)[0]


# Color mapping for segment types
segment_colors = {"braking": "red", "corner": "orange", "acceleration": "green"}

# Plot base track (gray)
fig.add_trace(
    go.Scattermapbox(
        lat=lat_arr,
        lon=lon_arr,
        mode="lines",
        line=dict(width=3, color="lightgray"),
        name="Track",
        showlegend=True,
    )
)

# Track which segment types we've added to legend
legend_added = {"braking": False, "corner": False, "acceleration": False}

# Plot all segments from the segments list
for seg in segments:
    indices = get_indices_for_range(seg.start_dist, seg.end_dist)
    if len(indices) > 0:
        color = segment_colors.get(seg.segment_type, "gray")
        show_in_legend = not legend_added[seg.segment_type]
        legend_added[seg.segment_type] = True

        legend_name = {
            "braking": "Braking Zone",
            "corner": "Corner",
            "acceleration": "Acceleration Zone",
        }.get(seg.segment_type, seg.segment_type)

        fig.add_trace(
            go.Scattermapbox(
                lat=lat_arr[indices],
                lon=lon_arr[indices],
                mode="lines",
                line=dict(width=6, color=color),
                name=legend_name if show_in_legend else None,
                showlegend=show_in_legend,
                legendgroup=seg.segment_type,
            )
        )

# Add corner apex markers with labels (for corner segments only)
for seg in segments:
    if seg.segment_type == "corner" and seg.apex_dist is not None:
        # Find index closest to apex distance
        apex_idx = int(np.argmin(np.abs(distance_arr - seg.apex_dist)))
        fig.add_trace(
            go.Scattermapbox(
                lat=[lat_arr[apex_idx]],
                lon=[lon_arr[apex_idx]],
                mode="markers+text",
                marker=dict(size=12, color="darkred", symbol="circle"),
                text=[seg.name],
                textposition="top right",
                textfont=dict(size=11, color="darkred"),
                name=None,
                showlegend=False,
            )
        )

fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=np.mean(lat_arr), lon=np.mean(lon_arr)),
        zoom=14,
    ),
    title="Track Segments: Braking (Red), Corner (Orange), Acceleration (Green)",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)"),
    width=900,
    height=700,
)

show_fig(fig)